# Session 8
# 6.1 Set estimator primitive options (such as resilience levels)
# 6.2 Understand the theoretical background behind the estimator primitive

27.02.2026

### Info

- [https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-estimator-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-estimator-options)
- [https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-twirling-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-twirling-options)
- [https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-resilience-options-v2](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-resilience-options-v2)
- [https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-zne-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-zne-options)
- [https://quantum.cloud.ibm.com/docs/en/guides/primitives](https://quantum.cloud.ibm.com/docs/en/guides/primitives)
- [https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques)


### Recap

Primitives are high-level computational building blocks designed to abstract away low-level details of quantum hardware interaction — similar to how high-level functions in classical programming shield you from CPU register manipulation.

| **Feature / Aspect** | **Estimator** | **Sampler** |
| --- | --- | --- |
| **Primary task** | Computes *expectation values* of observables for quantum states prepared by circuits | Samples measurement outcomes from quantum circuits |
| **What it processes** | Circuits + observables (+ optional parameters, precision) | Circuits (+ optional parameters, number of shots) |
| **Input PUB format** | `(circuit, observables, parameter_values?, precision?)` | `(circuit, parameter_values?, shots?)` |
| **Measurements in input circuit** | Explicit measurements are **ignored** | Circuits **must include measurements** |
| **Output type** | Expectation values and standard errors | Bitstring samples or counts distribution |
| **Typical result** | Real numbers (⟨observable⟩ values) | Counts dictionary or per-shot memory data |
| **Common use cases** | Variational algorithms (e.g., VQE cost function evaluation) | Sampling-based algorithms, probabilistic outputs |
| **Abstract base class (V2)** | `BaseEstimatorV2` | `BaseSamplerV2` |
| **Reference implementations** | `StatevectorEstimator`, `BackendEstimatorV2` | `StatevectorSampler`, `BackendSamplerV2`  |


## 6,1 Set estimator primitive options

## 6.2. Estimator primitive

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(simulator=False)

qc = QuantumCircuit(1)
qc.h(0)

# Define observable (Z on qubit 0)
observable = SparsePauliOp.from_list([("Z", 1.0)])
 
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")
result = job.result()
print(f">>> {result}")
print(f"  > Expectation value: {result[0].data.evs}")
print(f"  > Metadata: {result[0].metadata}")

qiskit_runtime_service.__init__:WARNING:2026-02-27 08:31:14,292: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: OneInstance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-27 08:31:15,461: Loading instance: OneInstance, plan: open
qiskit_runtime_service.backends:WARNING:2026-02-27 08:31:16,568: Using instance: OneInstance, plan: open


>>> Circuit ops (ISA): OrderedDict({'rz': 2, 'sx': 1})
>>> Job ID: d6gkghqthhns73912d80
>>> Job Status: QUEUED
>>> PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})
  > Expectation value: 0.015202381706467347
  > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {},

# Error Suppression and Mitigation

Quantum hardware today is noisy — physical qubits and gates introduce errors that distort computation outcomes.
Suppression and mitigation techniques aim to reduce or correct these errors without full quantum error correction.

In the context of primitives like Estimator, these techniques help produce more accurate expectation values from noisy quantum runs.

- **Error Suppression**: Techniques applied during circuit execution to prevent errors
- **Error Mitigation**: Post-processing techniques to reduce errors in results


## 🛠️ Error Suppression Techniques

### Dynamical Decoupling

**Key idea**: Reduce noise effects proactively at the hardware control level.

- Addresses coherent errors that accumulate when qubits idle between gates.
- Inserts carefully timed pulse sequences (identity operations) during idle periods to cancel unwanted interactions.
- Useful when there are gaps in a circuit; if qubits are continuously active, it may not help.

```python
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm" # 'XX', 'XpXm', 'XY4'
```


- "XX": use the sequence tau/2 - (+X) - tau - (+X) - tau/2
- XpXm": use the sequence tau/2 - (+X) - tau - (-X) - tau/2
- "XY4": use the sequence tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2


![Dynamical Decoupling](dd.png)

## Pauli Twirling (Randomized Compiling)

**Key idea**: Make noise statistics simpler so other mitigation techniques work better.

- Converts arbitrary noise into a simpler, structured Pauli channel by wrapping gates with randomly chosen Pauli operations that preserve the circuit’s ideal action.
- Replaces a single circuit with an ensemble of randomized variants.
- Coherent errors accumulate less severely after twirling, making noise behave more like random Pauli errors.


```python
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100
```

![twirling.png](twirling.png)

## 📊 Error Mitigation Strategies

These techniques don’t change the circuit itself but correct results or infer the noise-free answer.

### Twirled Readout Error Extinction (TREX)

**Key idea**: Mitigate classical noise in the final measurement phase.

- Focuses on measurement errors (incorrect qubit readouts).
- Uses randomization of measurement operators to diagonalize the error transfer matrix, which makes it easier to invert and correct classical biases in measurement outcomes.
- Requires calibration circuits to estimate the noise model.

```python
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100
```

![trex.png](trex.png)

### Zero-Noise Extrapolation (ZNE)

**Key idea**: Estimate the ideal output by fitting multiple noisy results to a noise-free limit.

- Approximate removal of noise by executing the same circuit under amplified noise levels and extrapolating the results back to the zero-noise limit.
- Noise amplification is typically done by gate folding (inserting gate + inverse pairs).
- Extrapolation models (e.g., linear, exponential) estimate what the expectation value would be absent noise.

```python
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"
```

![zne.png](zne.png)

## Probabilistic Error Amplification (PEA)

**Key idea**: Learn the actual noise and use that learned model for more reliable extrapolation.

- An enhancement of ZNE: first learns the noise model of entangling layers, then accurately amplifies noise for extrapolation.
- This two-step approach (learn + amplify + extrapolate) yields better accuracy than simple gate folding.

```python
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"
```

## Probabilistic Error Cancellation (PEC)

**Key idea**: Statistically cancel noise effects to recover unbiased results, paying a sampling overhead.

- Constructs an unbiased estimate of the ideal circuit by expressing the ideal operation as a linear combination of noisy circuits.
- Negative coefficients lead to a quasi-probability distribution and a sampling overhead that grows with circuit depth.
- Unlike ZNE, PEC theoretically eliminates bias but at a significant cost in the number of samples needed.

```python
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100
```



| Technique | Type | What It Does |
| --- | --- | --- |
| Dynamical Decoupling | Suppression     | Reduces coherent errors during idle  | 
| Pauli Twirling       | Suppression     | Converts arbitrary noise to Pauli noise        |
| TREX       | Mitigation      | Corrects measurement errors          |
| ZNE        | Mitigation      | Extrapolates to zero noise |
| PEA        | Mitigation (enhanced ZNE) | Learns noise and extrapolates better |
| PEC        | Mitigation      | Unbiased noise cancellation with sampling cost | ([quantum.cloud.ibm.com][1]) |

[1]: https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques "Error mitigation and suppression techniques | IBM Quantum Documentation"


## Estimator Options

[https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-estimator-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-estimator-options)

### Attributes

| **Option** | **Type** | **Description** | **Default** |
| --- | --- | --- | --- |
| `default_precision` | float | Default precision used for expectation value estimation when not specified in `run()` | 0.015625 (≈ 1/√4096) ([quantum.cloud.ibm.com][1]) |
| `default_shots` | int \| None | Total shots per circuit per configuration; if set, overrides `default_precision` | None ([quantum.cloud.ibm.com][1]) |
| `resilience_level` | int | High‑level resilience/error mitigation level for execution | 1 (minimally mitigated) ([quantum.cloud.ibm.com][1]) |
| `seed_estimator` | int | Seed for sampling control (randomization) | None ([quantum.cloud.ibm.com][1]) |
| `simulator` | dict / options | Simulator‑specific options (nested sub‑options) | † (default object) ([quantum.cloud.ibm.com][1]) |
| `dynamical_decoupling` | dict / options | Options for dynamical decoupling (error suppression) | † (default object) ([quantum.cloud.ibm.com][1])  |
| `twirling` | dict / options | Pauli twirling error mitigation configuration | † (default object) ([quantum.cloud.ibm.com][1])  |
| `resilience` | dict / options | Advanced resilience-specific settings (fine‑tuning mitigation)  | † (default object) ([quantum.cloud.ibm.com][1])  |
| `execution`  | dict / options | Execution environment controls (timeouts, priorities, etc.) | † (default object) ([quantum.cloud.ibm.com][1])  |
| `environment`  | dict / options | Environment options (logging, debug, etc.)  | † (default object) ([quantum.cloud.ibm.com][1])  |
| `max_execution_time` | int  | Maximum allowed execution time (seconds)  | `Unset` ([quantum.cloud.ibm.com][1]) |
| `experimental` | dict | Experimental options subject to change  | `Unset` ([quantum.cloud.ibm.com][1]) |

[1]: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-estimator-options "EstimatorOptions (latest version) | IBM Quantum Documentation"

### Methods

#### update

```python
from qiskit_ibm_runtime import EstimatorOptions as Options

options = Options()
options.update(
  default_shots=4096,
  max_execution_time=1200,
  resilience_level=2
)
```

Updates multiple options at once. This is useful for modifying options after creating an Estimator instance or for applying configuration changes.

In [33]:
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions as Options
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Example: Configuring Estimator Options

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a simple test circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
observable = SparsePauliOp("ZZ")

# Create Options object with custom settings
options = Options()

# Set basic execution parameters
options.max_execution_time = 300
options.default_shots = 2048 
options.default_precision = 0.01
options.seed_estimator = 12345

# Set resilience level (0=none, 1=basic, 2=moderate, 3=maximum)
options.resilience_level = 2  # moderate error mitigation

print(f"Max execution time: {options.max_execution_time} seconds")
print(f"Default shots: {options.default_shots}")
print(f"Default precision: {options.default_precision}")
print(f"Resilience level: {options.resilience_level}\n")

# Create Estimator with custom options
estimator = Estimator(mode=backend, options=options)

print("Updating Options\n")

# Create initial options
initial_options = Options()
initial_options.default_shots = 1024
initial_options.max_execution_time = 300

print(f"Initial shots: {initial_options.default_shots}")
print(f"Initial max time: {initial_options.max_execution_time}")

# Update options with new values
initial_options.update(default_shots=4096, max_execution_time=900)

print(f"Updated shots: {initial_options.default_shots}")
print(f"Updated max time: {initial_options.max_execution_time}\n")

qiskit_runtime_service.__init__:WARNING:2026-02-22 13:54:14,892: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: OneInstance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-22 13:54:15,423: Loading instance: OneInstance, plan: open
qiskit_runtime_service.backends:WARNING:2026-02-22 13:54:17,447: Using instance: OneInstance, plan: open


Max execution time: 300 seconds
Default shots: 2048
Default precision: 0.01
Resilience level: 2

Updating Options

Initial shots: 1024
Initial max time: 300
Updated shots: 4096
Updated max time: 900



## Twirling Options

[https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-twirling-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-twirling-options)

Twirling is a quantum error mitigation technique that intentionally randomizes certain circuit elements (like two‑qubit gates or measurements) using Pauli operators so that noise becomes more uniform and easier to average out statistically. This helps reduce bias from coherent errors by converting them into a mixture of simpler noise effects that cancel when averaged across many randomizations.

| **Option**  | **Type**  | **Description**  | **Default**  |
| --- | ---| ---| --- |
| `enable_gates`  | `bool`  | Whether **two‑qubit gate twirling** is enabled.  | `False` (disabled) ([IBM Quantum Documentation][1])  |
| `enable_measure`  | `bool`  | Whether **measurement twirling** is enabled. Applied to measurement instructions not in conditional logic. | `True` for *Estimator*, `False` for *Sampler* ([IBM Quantum Documentation][1]) |
| `num_randomizations`  | `int \| "auto"` | Number of random circuit variants to sample for twirling mitigation. If `"auto"`, the system chooses based on shot settings. | `"auto"` ([IBM Quantum Documentation][1])  |
| `shots_per_randomization` | `int \| "auto"` | Shots per random circuit sample. If `"auto"`, selected based on shots and mitigation type. | `"auto"` ([IBM Quantum Documentation][1])  |
| `strategy`  | `"active" \| "active-accum" \| "active-circuit" \| "all"` | Defines how qubits are twirled across layers of two‑qubit gates: |  |
| `active`, `active-circuit`, `active-accum`, or `all`. | `"active-accum"` ([IBM Quantum Documentation][1]) |  |  |

[1]: https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options "TwirlingOptions (latest version) | IBM Quantum Documentation"

strategy:
- If "active" only the instruction qubits in each individual twirled layer will be twirled.
- If "active-circuit" the union of all instruction qubits in the circuit will be twirled in each twirled layer.
- If "active-accum" the union of instructions qubits in the circuit up to the current twirled layer will be twirled in each individual twirled layer.
- If "all" all qubits in the input circuit will be twirled in each twirled layer.


![twirling_strategy_options](twirling_strategy_options.jpeg)


## Resilience Options

[https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-resilience-options-v2](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-resilience-options-v2)

Resilience options let you turn on and fine‑tune several advanced mitigation techniques, such as:
- Measurement error mitigation
- Zero‑Noise Extrapolation (ZNE)
- Probabilistic Error Cancellation (PEC)
- Layer‑specific noise learning

| **Option** | **Type** | **Description** | **Default** |
| --- | --- | --- | --- |
| `measure_mitigation` | `bool` / `Unset` | Enable measurement error mitigation. Controls whether measurement noise gets mitigated. | `Unset` (server uses default; typically *true*) ([quantum.cloud.ibm.com][1]) |
| `measure_noise_learning` | `MeasureNoiseLearningOptions` / dict | Fine‑tune how measurement noise is *learned* for mitigation. Provides sub‑options for calibration/noise learning. | Default object ([quantum.cloud.ibm.com][1])  |
| `zne_mitigation` | `bool` / `Unset` | Enable *Zero Noise Extrapolation* (ZNE) mitigation. ZNE runs circuits at amplified noise levels to extrapolate a low‑noise result.  | `Unset` (server default; typically *false*) ([quantum.cloud.ibm.com][1]) |
| `zne`  | `ZneOptions` / dict  | Additional configuration for ZNE (e.g., noise factors, extrapolation method). | Default object ([quantum.cloud.ibm.com][1])  |
| `pec_mitigation` | `bool` / `Unset` | Enable *Probabilistic Error Cancellation* (PEC). PEC builds a quasi‑probabilistic combination of noisy gates to cancel errors.  | `Unset` (server default; typically *false*) ([quantum.cloud.ibm.com][1]) |
| `pec`  | `PecOptions` / dict  | Detailed PEC tuning options (e.g., sampling budget, basis choices). | Default object ([quantum.cloud.ibm.com][1])  |
| `layer_noise_learning` | `LayerNoiseLearningOptions` / dict | Controls how the noise characteristics of circuit layers are *learned* for use in mitigation strategies.  | Default object ([quantum.cloud.ibm.com][1])  |
| `layer_noise_model`  | noise model / None / `Unset` | Optional pre‑learned noise model data. If provided, mitigation strategies that depend on noise data skip learning and use this model instead. | `Unset` (server default) ([quantum.cloud.ibm.com][1])  |

[1]: https://quantum.cloud.ibm.com/docs/api/qiskit-ibm-runtime/options-resilience-options-v2 "ResilienceOptionsV2 (latest version) | IBM Quantum Documentation"


In [34]:
from qiskit_ibm_runtime import EstimatorOptions as Options
from qiskit_ibm_runtime.options import MeasureNoiseLearningOptions,LayerNoiseLearningOptions,ZneOptions

# Example: Configuring Resilience Options

# Method 1: Using resilience_level
levels = [0, 1, 2]
descriptions = [
  "No error mitigation",
  "Minimal measurement error mitigation",
  "Medium mitigation"
]

print("Method 1: Using resilience_level")
for level, desc in zip(levels, descriptions):
  options = Options()
  options.resilience_level = level
  print(f"Level {level}: {desc}")

print("\nExamples:")
# Minimal resilience
print("Minimal resilience")
quick_test = Options()
quick_test.resilience_level = 0
quick_test.default_shots = 256
print(f"  Resilience: {quick_test.resilience_level} (none)")
print(f"  Shots: {quick_test.default_shots}")

# Moderate resilience
print("Moderate resilience")
moderate = Options()
moderate.resilience_level = 1
moderate.default_shots = 1024
print(f"  Resilience: {moderate.resilience_level} (basic)")
print(f"  Shots: {moderate.default_shots}")

# High resilience
print("High resilience")
high = Options()
high.resilience_level = 2
high.default_shots = 4096
print(f"  Resilience: {high.resilience_level} (advanced)")
print(f"  Shots: {high.default_shots}")
  
# Method 2: Control with Resilience Options

options = Options()

# Configure individual resilience techniques
# Basic readout correction
options.resilience.measure_mitigation = True
# Learn measurement noise
options.resilience.measure_noise_learning = MeasureNoiseLearningOptions()
# Enable Zero-Noise Extrapolation
options.resilience.zne_mitigation = True
# Disable PEC (computationally expensive)
options.resilience.pec_mitigation = False
# Learn layer-dependent noise
options.resilience.layer_noise_learning = LayerNoiseLearningOptions() 

# Configure ZNE specifically
options.resilience.zne = ZneOptions(noise_factors=[1.0, 2.0, 3.0], extrapolator="exponential")

print("\nMethod 2: Control with Resilience Options")
print("configured resilience settings:")
print(f"  Measurement mitigation: {options.resilience.measure_mitigation}")
print(f"  ZNE mitigation: {options.resilience.zne_mitigation}")
print(f"  PEC mitigation: {options.resilience.pec_mitigation}")
print(f"  ZNE noise factors: {options.resilience.zne.noise_factors}")
print(f"  ZNE extrapolator: {options.resilience.zne.extrapolator}\n")


Method 1: Using resilience_level
Level 0: No error mitigation
Level 1: Minimal measurement error mitigation
Level 2: Medium mitigation

Examples:
Minimal resilience
  Resilience: 0 (none)
  Shots: 256
Moderate resilience
  Resilience: 1 (basic)
  Shots: 1024
High resilience
  Resilience: 2 (advanced)
  Shots: 4096

Method 2: Control with Resilience Options
configured resilience settings:
  Measurement mitigation: True
  ZNE mitigation: True
  PEC mitigation: False
  ZNE noise factors: [1.0, 2.0, 3.0]
  ZNE extrapolator: exponential



## ZNE Options

[https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-zne-options](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-zne-options)

Instead of performing full-fledged error correction, ZNE intentionally amplifies the noise in controlled ways and then mathematically extrapolates the measured expectation values back to what they would be in the zero‑noise limit. This helps estimate a more accurate (less noisy) result from noisy hardware than a raw measurement would give

ZNE consists of two main parts:
- Noise amplification – modify or fold gates to increase effective noise.
- Extrapolation – fit a curve over measured data at different noise levels and estimate the ideal (zero noise) value.

| **Option** | **Type**  | **Description**  | **Default** |
| --- | --- | --- | --- |
| `amplifier`  | `str` | unset | Strategy used to **amplify noise** in circuits. Choices include:<br>• `"gate_folding"` – amplify noise by folding two‑qubit gates;<br>• `"gate_folding_front"` / `"gate_folding_back"` – controlled placement of folds;<br>• `"pea"` – use **Probabilistic Error Amplification (PEA)**, a learned noise model to amplify noise more accurately.  | `Unset` (server default); usually `"gate_folding"` for classic ZNE ([quantum.cloud.ibm.com][1]) |
| `noise_factors`  | `Sequence[float]` | unset | List of **noise amplification factors** to use. Each factor represents how much noise to scale relative to the base circuit before extrapolating (e.g., `[1, 3, 5]` means run circuits at 1×, 3×, and 5× noise). | `Unset` (server may default to common sets like `(1,3,5)`) ([quantum.cloud.ibm.com][1]) |
| `extrapolator` | `str` | sequence | unset  | Functional form(s) used to **fit and extrapolate** the noisy results to zero noise. Options include:<br>• `"linear"` – linear fit;<br>• `"exponential"` – exponential decay fit;<br>• `"double_exponential"` – sum of two exponentials;<br>• `"polynomial_degree_k"` – polynomial of degree *k*;<br>• `"fallback"` – no extrapolation, return lowest‑noise result. | `Unset` (server default often includes a set like `("exponential", "linear")`) ([quantum.cloud.ibm.com][1]) |
| `extrapolated_noise_factors` | `Sequence[float]` | unset | Additional noise factor values at which to **evaluate the fitted extrapolation model** for reporting (i.e., where the extrapolated curve is sampled). Affects **reporting**, not execution.  | `Unset` (server selects default) ([quantum.cloud.ibm.com][1]) |

[1]: https://quantum.cloud.ibm.com/docs/api/qiskit-ibm-runtime/options-zne-options?utm_source=chatgpt.com "ZneOptions (latest version) | IBM Quantum Documentation"


In [35]:
# Example: Configuring ZNE Options

print("=== Zero-Noise Extrapolation (ZNE) Configuration ===\n")

# First, enable ZNE at the resilience level
options = Options()
options.resilience.zne_mitigation = True

# Configure ZNE parameters
print("ZNE Configuration Examples:\n")

# Basic ZNE with gate folding
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 2.0, 3.0]
options.resilience.zne.extrapolator = "linear"

print(f"Case 1:")
print(f"  Amplifier: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  Extrapolator: {options.resilience.zne.extrapolator}")

# Advanced ZNE for better accuracy
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "exponential"

print(f"Case 2:")
print(f"  Amplifier: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  Extrapolator: {options.resilience.zne.extrapolator}")

# Create ZNE configuration
zne_config = Options()

# Enable and configure ZNE
zne_config.resilience.zne_mitigation = True
zne_config.resilience.zne.amplifier = "gate_folding"
zne_config.resilience.zne.noise_factors = [1.0, 2.0, 3.0, 4.0]
zne_config.resilience.zne.extrapolator = "polynomial_degree_1"  # Polynomial extrapolation


# Set shots and execution time for ZNE
zne_config.default_shots = 2048
zne_config.max_execution_time = 600  # 10 minutes

print("Complete ZNE Configuration:")
print(f"  ZNE enabled: {zne_config.resilience.zne_mitigation}")
print(f"  Amplifier: {zne_config.resilience.zne.amplifier}")
print(f"  Noise factors: {zne_config.resilience.zne.noise_factors}")
print(f"  Extrapolator: {zne_config.resilience.zne.extrapolator}")
print(f"  Shots per factor: {zne_config.default_shots}")
print(f"  Total estimated shots: {zne_config.default_shots * len(zne_config.resilience.zne.noise_factors)}")
print(f"  Max execution time: {zne_config.max_execution_time} seconds")

=== Zero-Noise Extrapolation (ZNE) Configuration ===

ZNE Configuration Examples:

Case 1:
  Amplifier: gate_folding
  Noise factors: [1.0, 2.0, 3.0]
  Extrapolator: linear
Case 2:
  Amplifier: gate_folding
  Noise factors: [1.0, 1.5, 2.0, 3.0, 4.0]
  Extrapolator: exponential
Complete ZNE Configuration:
  ZNE enabled: True
  Amplifier: gate_folding
  Noise factors: [1.0, 2.0, 3.0, 4.0]
  Extrapolator: polynomial_degree_1
  Shots per factor: 2048
  Total estimated shots: 8192
  Max execution time: 600 seconds


In [36]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions as Options
from qiskit_ibm_runtime.options import ResilienceOptionsV2, ZneOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a simple test circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
observable = SparsePauliOp("ZZ")

# Create Options object with custom settings
options = Options()

# Configure ZNE
zne_opts = ZneOptions(
    amplifier="gate_folding",
    noise_factors=[1.0, 3.0, 5.0],
    extrapolator="linear"
)
# Include ZNE in resilience options
resilience_opts = ResilienceOptionsV2(
    zne_mitigation=True,
    zne=zne_opts
)

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

estimator = Estimator(mode=backend)
estimator.options.resilience = resilience_opts
job = estimator.run([(isa_circuit, isa_observable)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")
result = job.result()
print(f">>> {result}")
print(f"  > Expectation value: {result[0].data.evs}")
print(f"  > Metadata: {result[0].metadata}")


qiskit_runtime_service.__init__:WARNING:2026-02-22 13:54:19,694: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: OneInstance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-22 13:54:20,321: Loading instance: OneInstance, plan: open
qiskit_runtime_service.backends:WARNING:2026-02-22 13:54:22,765: Using instance: OneInstance, plan: open


>>> Circuit ops (ISA): OrderedDict({'rz': 6, 'sx': 3, 'cz': 1})
>>> Job ID: d6dfp04nsg9c739bb4bg
>>> Job Status: QUEUED
>>> PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), evs_noise_factors=np.ndarray(<shape=(3,), dtype=float64>), stds_noise_factors=np.ndarray(<shape=(3,), dtype=float64>), ensemble_stds_noise_factors=np.ndarray(<shape=(3,), dtype=float64>), evs_extrapolated=np.ndarray(<shape=(1, 4), dtype=float64>), stds_extrapolated=np.ndarray(<shape=(1, 4), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {'zne': {'extrapolator': 'linear'}}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_

---

## Questions

**1) Which option is an example of error suppression, not error mitigation?**

A) Zero-noise extrapolation (ZNE)

B) Measurement error mitigation

C) Dynamical decoupling

D) Probabilistic error cancellation

**Answer:**
<details> <br/>
C) Dynamical decoupling as it reduces error during execution


</details>

**2) Which circuit would benefit more from dynamical decoupling and why?**

```python
# Circuit A:
qc_a = QuantumCircuit(5)
qc_a.h(0)
qc_a.cx(0, 1)
# Qubits 2, 3, 4 remain idle
qc_a.measure_all()

# Circuit B:
qc_b = QuantumCircuit(5)
qc_b.h(range(5))
for i in range(4):
    qc_b.cx(i, i+1)
qc_b.barrier()
for i in range(4, 0, -1):
    qc_b.cx(i, i-1)
qc_b.measure_all()
```

A) Both equally, as all circuits benefit from dynamical decoupling

B) Circuit A, because it has idle qubits where pulse sequences can be inserted

C) Circuit B, because dense circuits need more error suppression

D) Neither, as dynamical decoupling only works on simulators

**Answer:**
<details> <br/>
B) Circuit A

Dyanamical decoupling is more useful with circuits that has idle qubits


</details>

**3) Which configuration is correct for using PEA?**

```python
A) estimator.options.resilience.pec_mitigation = True
   estimator.options.resilience.pec.amplifier = "pea"

B) estimator.options.resilience.zne_mitigation = True
   estimator.options.resilience.zne.amplifier = "pea"

C) estimator.options.zne.enable_gates = True
   estimator.options.zne.amplifier = "pea"

D) estimator.options.dynamical_decoupling.enable = True
   estimator.options.dynamical_decoupling.amplifier = "pea"
```

**Answer:**
<details> <br/>
B) use zne_mitigation = True

PEA is a noise amplification method for ZNE, so ZNE must be enabled.and the correct syntax to enable is `estimator.options.resilience.zne_mitigation = True`

</details>

**4) Why does Probabilistic Error Cancellation (PEC) produce a greater overhead on the circuit time?**

A) Because it requires quantum error correction codes that need many physical qubits

B) Because it requires to run the circuit in session mode

C) Because it needs to run the circuitw millions of times to get accurate statistics

D) Because the sampling overhead grows exponentially with circuit depth

**Answer:**
<details> <br/>

D) Because the sampling overhead grows exponentially with circuit depth

PEC expresses the target circuit as a linear combination of noisy circuits forming a quasi-probability distribution. The overhead grows exponentially with the circuit depth


</details>

**5) What is the primary purpose of setting resilience_level in EstimatorOptions?**

A) To control circuit execution on the backend.

B) To select the transpilation resilience optimization level.

C) To enable and coordinate error mitigation techniques.

D) To configure batching and session behavior.

**Answer:**
<details> <br/>

C) `resilience_level` determines which error mitigation strategies (twirling, ZNE, ...) are enabled and how they are applied.

</details>

**6) What is the key difference between active and active-circuit twirling strategies?**

A) active twirls measurements, active-circuit does not twirl measurments

B) active applies twirling around instructions on each layer separtely, while active-circuit wraps instruction qubits in each layer

C) active accumulates twirling across layers, while active-circuit resets after each layer

D) active is only supported on simulators

**Answer:**
<details> <br/>

B) when using 'active' only the instruction qubits in each individual twirled layer will be twirled, while using 'active-circuit' twirls the union of all instruction qubits in the circuit in each twirled layer.

</details>

**7) Complete this code to configure ZNE with 5 noise factors and polynomial extrapolation:**

```
from qiskit_ibm_runtime import EstimatorOptions as Options

options = Options()

# Enable ZNE mitigation
options.resilience.______ = True

# Configure ZNE parameters
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.______ = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "______"
```

Which option correctly fills all three blanks?

A) zne_enable, noise_levels, polynomial

B) zne, factors, polynomial_degree_2

C) zne_mitigation, noise_factors, polynomial_degree_2

D) enable_zne, noise_factors, poly_2

**Answer:**
<details> <br/>

C) Correct attributes names are `zne_mitigation`, `noise_factors`, `polynomial_degree_2`
</details>

**8) Which call correctly submits work to Estimator V2 using the Primitive Unified Bloc (PUB) tuple?**

A) `job = estimator.run([(isa_circ, isa_obs, param_values)])`

B) `job = estimator.run([(isa_obs, isa_circ, param_values)])`

C) `job = estimator.run([(isa_circ, param_values)])`

D) `job = estimator.run((isa_circ, isa_obs, param_values))`

**Answer:**
<details> <br/>

A) The Estimator expects a sequence of PUB tuples in the form (circuit, observable, parameter_values); the example shows estimator.run([(isa_circuit, isa_observable, param_values)]).
</details>

**9) Which initialization correctly selects the execution mode consistent with the examples?**

A) 
```
from qiskit_ibm_runtime import EstimatorV2 as Estimator
estimator = Estimator(mode=backend)
```
B) 
```
from qiskit_ibm_runtime import EstimatorV2 as Estimator
estimator = Estimator(mode="session")  # string literal
```
C)
```
from qiskit_ibm_runtime import EstimatorV2 as Estimator
estimator = Estimator()  # mode is mandatory
```
D)
```
from qiskit_ibm_runtime import EstimatorV2 as Estimator
estimator = Estimator(mode="batch").run(backend)
```

**Answer:**
<details> <br/>

A) When you initialize the Estimator, use the mode parameter to specify the mode you want it to run in. Possible values are batch, session, or backend objects for batch, session, and job execution mode, respectively.
</details>




**10) In EstimatorV2, what does options.default_precision control and what is its default value?**

A) It sets the target standard error for expectation values (used when shots aren’t given); default is 0.015625 (1/√4096).

B) It sets the transpiler optimization level; default is 1.

C) It fixes the number of shots for each circuit configuration; default is 4096.

D) It controls backend run priority; default is “normal”.

**Answer:**
<details> <br/>

The answer is a.

default_precision defines the default statistical precision target for expectation values when neither the PUB nor run() specifies precision; EstimatorV2 chooses shots to meet this. The documented default is 0.015625 (i.e., 1/√4096).

</details>

**11) Regarding options.default_shots in EstimatorV2, which statement is correct?**

A) If set, it overrides default_precision, and when twirling is enabled it is split across randomizations.

B) It is ignored whenever twirling is enabled.

C) It only applies to simulators and is ignored on hardware.

D) It sets the maximum number of shots per job but never overrides precision.

**Answer:**
<details> <br/>

The answer is a.

When default_shots is provided, it takes precedence over default_precision. If Pauli twirling is active, the total shots are divided among the circuit randomizations according to the twirling options.

</details>

**12) Which mapping of resilience_level to behavior is correct for EstimatorV2?**

A) 0: No mitigation; 1: Readout mitigation + measurement twirling; 2: Adds bias‑reducing estimator mitigation (medium cost).

B) 0: Readout mitigation only; 1: Full error correction; 2: No mitigation.

C) 0: Measurement twirling; 1: Probabilistic error cancellation; 2: Zero‑noise extrapolation only.

D) 0: No mitigation; 1: Full QEC; 2: Hardware calibration reset only.

**Answer:**
<details> <br/>

The answer is a.

resilience_level increases mitigation complexity:

    0 disables mitigation
    1 applies minimal‑cost techniques like readout mitigation (and measurement twirling in V2)
    2 adds medium‑cost estimator bias‑reduction methods

</details>

**13) For TwirlingOptions.strategy, what best describes the default "active-accum" behavior?**

A) In each twirled layer, twirl the union of instruction qubits encountered up to that layer.

B) Twirl only the instruction qubits of the current layer, ignoring earlier ones.

C) Twirl all qubits in the circuit at every twirled layer.

D) Twirl only the first two qubits of the circuit to minimize overhead.


**Answer:**
<details> <br/>

The answer is a.

"active-accum" accumulates the set of instruction qubits seen so far and twirls that growing set at each subsequent twirled layer, which is default strategy.

</details>

**14) What is the default of TwirlingOptions.enable_measure and where does it apply?**

A) True for Estimator, False for Sampler; enables twirling on measurement instructions not inside conditionals.

B) True for both Estimator and Sampler; always twirl all measurements.

C) False for both primitives; measurement twirling must be enabled via resilience_level only.

D) True only on simulators; False on hardware backends.

**Answer:**
<details> <br/>

The answer is a.

Measurement twirling is enabled by default for Estimator and disabled by default for Sampler. It applies to measurement operations provided they are not within a conditional block.

</details>